### Our custom Algorithm
#### For our reference - we can include some in the report and/or presentation
The file contains the following:

- Cleaning functions (same as the ones I added for data preprocessing)
- Training Functions: these are used once to train the algorithm and extract the csv files to be used for the detection
- Detection functions: these utilize the csv files generated fromt eh training to classify the input text

I included as many comments as I felt necessary but let me know if you guys think there's something that's unclear.


In [ ]:
import re
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk import bigrams, trigrams
from nltk.probability import FreqDist
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


#-------------------   cleaning functions (copied from our data preprocessing)  -------------------

# Lowercasing funcction - simple but we will probably need it for our custom algorithm (similarity search)
def lowercase(text):
    return text.lower()

# a function for removing stop words
def remove_stop_words(text):
    # generate tokenized words from the text - NLTK will miss things up without tokens
    text = nltk.word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    celaned_text = []
    for word in text:
        if word not in stop_words:
            celaned_text.append(word)
    # converting the list to a string so we can replace the actual text with the cleaned text
    return ' '.join(celaned_text)


# function to extract n-grams from the text
def extract_ngrams(text, n_gram, length=None, remove_punct=False):
    # returns the most common n-grams in the text as a list
    # n_gram can be 1 for unigrams, 2 for bigrams, and 3 for trigrams ...etc.
    # length is the number of n-grams to return, default is None which returns all
    tokens = nltk.word_tokenize(lowercase(text))
    # keeps only words, letters and numbers
    if remove_punct:
        tokens = [word for word in tokens if word.isalnum()]

    ngrams = nltk.ngrams(tokens, n_gram)
    # full dictionary of n-grams and their frequencies
    ngram_frequency = FreqDist(ngrams)
    # returning only the top number based on the length parameter above
    # if length is None, return all n-grams
    return ngram_frequency.most_common(length)
    

# function to remove punctuation from the text
def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

#-------------------   training functions   -------------------

#extracting most used words in texts
def most_used_by(text_list, filter_top = 100):
    words_count = {}
    for text in text_list:
        cleaned_text = remove_stop_words(lowercase(text))
        cleaned_text = remove_punctuation(cleaned_text)
        words = cleaned_text.split()
        # counting and storing the frequency of each word
        for word in words:
            if word in words_count:
                words_count[word] += 1
            else:
                words_count[word] = 1
    # sort the words by frequency and return the top ones
    sorted_words = sorted(words_count.items(), key=lambda item: item[1], reverse=True)
    return sorted_words[:filter_top]

# most common n-grams
# similar to the one above but focuses on n-grams instead of single words
# I think if we use 1-gram it could output the same think but I did not realize that until I implemented both

def most_used_ngrams_by(text_list, n_gram=2, filter_top=100):
    ngrams_count = {}
    for text in text_list:
        # using the n-gram functions from above
        text_ngrams = extract_ngrams(text, n_gram, length=None, remove_punct=True)
        # counting n-gram frequency
        for ngram, count in text_ngrams:
            if ngram in ngrams_count:
                ngrams_count[ngram] += count
            else:
                ngrams_count[ngram] = count
    # sorting before we return it
    sorted_ngrams = sorted(ngrams_count.items(), key=lambda item: item[1], reverse=True)
    return sorted_ngrams[:filter_top]
    

def calculate_ai_word_ratios(ai_texts, human_texts, filter_top_n=1000, return_top_n = 100):
    # I realized that I made these functions retun lists of tuples instead of dictionaries
    #  so instead of rewriting them I decided to add dctionary conversion here
    
    ai_words    = dict(most_used_by(ai_texts, filter_top=filter_top_n))
    human_words = dict(most_used_by(human_texts, filter_top=filter_top_n))

    # total words in each group for getting a rate instead of count
    total_ai    = sum(ai_words.values())
    total_human = sum(human_words.values())

    results = []
    for word, ai_freq in ai_words.items():

        # getting a rate instead of count
        ai_rate    = ai_freq / total_ai * filter_top_n
        human_rate = human_words.get(word, 0) / total_human * filter_top_n

        # raatio of how many times more common is this word in AI vs human 
        ratio = ai_rate / (human_rate + 1) # +1 avoids division by 0 when the word is not present in human texts

        results.append((word, ai_rate, human_rate, ratio))

    # sorting by ratio which is the 4th item in the tuple
    results.sort(key=lambda item: item[3], reverse=True)
    return results[:return_top_n]

# similar to the one above but for n-grams instead of single words
def calculate_ai_ngrams_ratios(ai_texts, human_texts, n_gram=2,filter_top_n=1000, return_top_n = 100):
    
    ai_ngrams    = dict(most_used_ngrams_by(ai_texts,    n_gram, filter_top=filter_top_n))
    human_ngrams = dict(most_used_ngrams_by(human_texts, n_gram, filter_top=filter_top_n))
    total_ai    = sum(ai_ngrams.values())
    total_human = sum(human_ngrams.values())

    results = []
    for ngram, ai_freq in ai_ngrams.items():
        ai_rate    = ai_freq / total_ai * filter_top_n
        human_rate = human_ngrams.get(ngram, 0) / total_human * filter_top_n
        ratio = ai_rate / (human_rate + 1)
        results.append((ngram, ai_rate, human_rate, ratio))
    results.sort(key=lambda item: item[3], reverse=True)
    return results[:return_top_n]




# I tesed these and they turned out to be insignificant signals in our dataset 
# so I removed them from the final algorithm but I am keeping the code here for reference

# similarity search function using cosine similarity
# https://www.geeksforgeeks.org/machine-learning/python-measure-similarity-between-two-sentences-using-cosine-similarity/

def sentence_similarity_score(text):
    # split the text into sentences using nltk's sentence tokenizer
    sentences = nltk.sent_tokenize(text)
    #if we have only once sentence
    if len(sentences) < 2:
        return 0.0

    tfidf_matrix = TfidfVectorizer().fit_transform(sentences)
    similarity_matrix = cosine_similarity(tfidf_matrix)

    total_similarity = 0
    number_of_pairs = 0
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            total_similarity += similarity_matrix[i][j]
            number_of_pairs += 1
    return round(total_similarity / number_of_pairs, 4)


def passive_voice_score(text):
    # lowercase the text before processing
    text = lowercase(text)
    sentences = nltk.sent_tokenize(text)
    # to avoid errors if no text was passed
    if not sentences:
        return 0.0
    # passive voice = a form of to be followed by a past tense verb (ending in "ed") like "is required", "are expected", "will be reviewed"
    # I am using simple regex to look for this pattern
    passive_count = 0
    for sentence in sentences:
        # looks for one verb from the list followed by one or more spaces followed by a word ending in ed
        if re.search(r'\b(is|are|was|were|be|been|being)\s+\w+ed\b', sentence):
            passive_count += 1
    return round(passive_count / len(sentences), 4)

# conrtactions to detect human text as human tend to use it more based on the little research I did on this topic
def contraction_score(text):
    text = lowercase(text)
    sentences = nltk.sent_tokenize(text)
    if not sentences:
        return 0.0

    # count how many sentences contain a contraction like you'll, we're, don't, etc.
    contraction_count = 0
    for sentence in sentences:
        # within a word, search for one or more charecters followed by ' followed by one or more characters
        if re.search(r"\b\w+'\w+\b", sentence):
            contraction_count += 1

    # more contractions = more human-like
    # so we invert the score — fewer contractions = higher AI score
    contraction_rate = contraction_count / len(sentences)
    return round(1- contraction_rate, 4)


#-------------------   detection functions   -------------------
# compare input against stored AI words

def score_against_ai_words(text, ai_words):
    cleaned = remove_punctuation(remove_stop_words(lowercase(text)))
    words   = cleaned.split()
    score = 0.0
    # to avoid errors if no text was passed
    if not words:
        return round(score, 4)
    # for each word in the input, look up its AI ratio
    total_ratio = 0
    for word in words:
        if word in ai_words:
            total_ratio += ai_words[word]
    score = total_ratio / len(words)
    
    return round(score, 4)

# again, this is similar to the one above but for n-grams instead of single words
def score_against_ai_ngrams(text, ai_ngrams, n_gram=2):
    tokens = nltk.word_tokenize(lowercase(text))
    tokens = [word for word in tokens if word.isalnum()]
    text_ngrams = list(nltk.ngrams(tokens, n_gram))
    score = 0.0
    if not text_ngrams:
        return round(score, 4)
    total_ratio = 0
    for ngram in text_ngrams:
        if ngram in ai_ngrams:
            total_ratio += ai_ngrams[ngram]
    score = total_ratio / len(text_ngrams)

    return round(score, 4)


# -------------------  detecting part --------------------------

def detect_ai(text, ai_words, ai_bigrams, ai_trigrams,threshold=0.2):
    # score the text against each dataset
    word_score     = score_against_ai_words(text, ai_words)
    bigram_score   = score_against_ai_ngrams(text, ai_bigrams,  n_gram=2)
    trigram_score  = score_against_ai_ngrams(text, ai_trigrams, n_gram=3)

    # assigning weights to each score
    final_score = (word_score    * 0.50 +
                   bigram_score  * 0.30 +
                   trigram_score * 0.20)

    # we can add the scores from other functions for other datasets but for this one they did not work
    final_score = round(final_score, 4)
    classification = "AI" if final_score > threshold else "Human"
    return final_score, classification

In [119]:
# training the algorithm on the jobs dataset - I ran these and generated the csv and pushed them to the repo
# we will need to run them again if we update our training dataset though
# we do not need this in the streamlit part

df = pd.read_csv('combined.csv')

# split the dataset into ai and human texts
ai_texts    = df[df["is_AI"] == 1]["job_summary"].tolist()
human_texts = df[df["is_AI"] == 0]["job_summary"].tolist()

# training the algorithm and creating the csv files
# words
word_results = calculate_ai_word_ratios(ai_texts, human_texts,return_top_n=500)
ai_words = {}
for word, ai_rate, human_rate, ratio in word_results:
    ai_words[word] = ratio

word_df = pd.DataFrame(list(ai_words.items()), columns=['word', 'ratio'])
word_df.to_csv('ai_words.csv', index=False)
# bigrams
bigram_results = calculate_ai_ngrams_ratios(ai_texts, human_texts, n_gram=2,return_top_n=500)
ai_bigrams = {}
for ngram, ai_rate, human_rate, ratio in bigram_results:
    ai_bigrams[ngram] = ratio

bigram_data = []
for ngram, ratio in ai_bigrams.items():
    bigram_str = ' '.join(ngram)
    bigram_data.append([bigram_str, ratio])
bigram_df = pd.DataFrame(bigram_data, columns=['bigram', 'ratio'])
bigram_df.to_csv('ai_bigrams.csv', index=False)

# trigrams
trigram_results = calculate_ai_ngrams_ratios(ai_texts, human_texts, n_gram=3,return_top_n=500)
ai_trigrams = {}
for ngram, ai_rate, human_rate, ratio in trigram_results:
    ai_trigrams[ngram] = ratio

trigram_data = []
for ngram, ratio in ai_trigrams.items():
    trigram_str = ' '.join(ngram)
    trigram_data.append([trigram_str, ratio])
trigram_df = pd.DataFrame(trigram_data, columns=['trigram', 'ratio'])
trigram_df.to_csv('ai_trigrams.csv', index=False)

print("training done")



training done


In [ ]:
# testing  detect_ai on the dataset to determine the threshold that we need to set

ai_scores = []
ai_classifications = []

for i in range(len(df)):
    job_text = df["job_summary"][i]
    result = detect_ai(job_text, ai_words, ai_bigrams, ai_trigrams, threshold=0.25)
    
    # extracting score and classification from the result tuple and appending to the lists
    score = result[0]
    classification = result[1]
    ai_scores.append(score)
    ai_classifications.append(classification)

# add both columns to the dataframe to help analayze and calculate error and accuracy
df["algorithm_score"] = ai_scores
df["algorithm_classification"] = ai_classifications



# calculating error in the detection compared to the target column
incorrect_detections = 0
for i in range(len(df)):
    actual = "AI" if df["is_AI"][i] == 1 else "Human"
    predicted = df["algorithm_classification"][i] 
    if actual != predicted:
        incorrect_detections += 1

print("number of incorrect detections:", incorrect_detections)
print("accuracy:", round((len(df) - incorrect_detections) / len(df), 4))
df.head(15)



number of incorrect detections: 371
accuracy: 0.8763


,job_title,job_location,job_salary,job_summary,is_AI,source,algorithm_score,algorithm_classification
0,Data Science Manager,"Chicago, IL","$120,000 - $150,000 a year",Our client is hiring a Data Scientist to suppo...,1,chatgpt ai,0.1162,Human
1,AI Engineer,"Boston, MA","$120,000 - $150,000 a year",This role sits within a cross-functional analy...,1,chatgpt ai,0.1035,Human
2,Statistician - Behavioral Biology,"Silver Spring, MD 20910",NaN,Overview: About The Geneva Foundation ...,0,human,0.0993,Human
3,Data Science Lead,"Chicago, IL 60601",NaN,Pinnacle AI Labs is a world-class research and...,1,claude ai,0.3204,AI
4,Data Scientist,"Boston, MA 02124",NaN,The Opportunity MassMutual’s Advanced Analy...,0,human,0.2476,Human
5,Data Engineer,"Plano, TX",NaN,Overview PepsiCo operates in an environme...,0,human,0.2425,Human
6,Senior Data Science Analyst,"Phoenix, AZ 85004","$94,000 - $132,000 a year",Permanent opportunity with a FULL suite of b...,0,human,0.2485,Human
7,Cloud Data Engineer II - Enterprise Analytics ...,"Hartford, CT","$113,900 - $188,000 a year",...,0,human,0.1663,Human
8,AI Product Manager,"Boston, MA 02119",NaN,Overview: As a national leader in the co...,0,human,0.1782,Human
9,Data Scientist,"Seattle, WA","$120,000 - $150,000 a year",We’re looking for someone to join our growing ...,1,chatgpt ai,0.1269,Human


In [121]:

max_AI_score = 0
max_human_score = 0
for i in range(len(df)):
    if df["algorithm_classification"][i] == "Human":
        if df["algorithm_score"][i] > max_human_score:
            max_human_score = df["algorithm_score"][i]
    if df["algorithm_classification"][i] == "AI":
        if df["algorithm_score"][i] > max_AI_score:
            max_AI_score = df["algorithm_score"][i]

print("max score for human text:", max_human_score)      #<-- we can reference this to play with the threshold
print("max score for AI text:", max_AI_score)

max score for human text: 0.2497
max score for AI text: 2.7738


In [ ]:
# this is from chatgpt

input = """Pinnacle AI Labs is a world-class research and applied AI organization at the forefront of machine learning innovation.  We are looking for a Data Science Lead who is eager to work on challenging, high-impact projects in a supportive team environment.

At Pinnacle AI Labs, we believe that the best work happens when talented people are given the freedom to solve hard problems in creative ways.  Our culture is built on a foundation of trust, transparency, and a shared commitment to excellence.  We invest deeply in our people, offering robust professional development programs, mentorship opportunities, and a collaborative environment where every voice is heard and valued.  We are proud to have assembled a team of world-class data scientists, engineers, and business professionals who are passionate about using data to create meaningful change.

Position Overview:
The Data Science Lead will work at the intersection of data science and software engineering, developing robust, production-ready machine learning systems.  You will be expected to contribute to all phases of the data science lifecycle, from problem formulation and data collection through model deployment and monitoring.

As our data needs continue to grow, this role represents a unique opportunity to build and shape our analytical capabilities from the ground up.  You will have the rare chance to influence not only the technical architecture of our data systems but also the culture and practices around data-driven decision making at Pinnacle AI Labs.  This is the kind of role that talented data professionals spend their careers looking for — high impact, high visibility, and high reward.

The duties and responsibilities stated are a general summary and not all inclusive.

Core Responsibilities:
  Create and maintain interactive dashboards and reports that empower business stakeholders to make informed decisions.
  Build real-time and batch inference systems for model serving at scale, with a focus on reliability and low latency.
  Work with business leaders to scope, plan, and execute data science projects from inception through delivery.
  Evaluate and experiment with new algorithms, frameworks, and tools to continuously improve model accuracy and performance.
  Oversee data integrity and governance, working with compliance and audit teams to identify discrepancies and ensure data hygiene.
  Contribute to the development of internal tools and platforms to accelerate data science workflows and improve team productivity.
  Research and implement state-of-the-art natural language processing and computer vision techniques.
  Contribute to the hiring and onboarding of new data science team members, helping to build a world-class team.
  Drive adoption of data products and analytical tools across the organization through training and change management.
  Collaborate with engineering, product, and business teams to translate ambiguous requirements into clear, impactful technical solutions.
  Lead end-to-end machine learning projects, from problem definition and data collection through model deployment and ongoing monitoring.

  All other duties as assigned.

Minimum Requirements:
Educational Background:
Bachelor's degree required; Master's degree or PhD in Computer Science, Applied Mathematics, Statistics, or a related field preferred.

Experience/Skill:
Required:
  Demonstrated experience with MLOps practices including model versioning, CI/CD pipelines, and production monitoring.
  Experience designing, executing, and analyzing A/B tests and other controlled experiments in a business context.
  Strong proficiency in Python, including core data science libraries such as Pandas, NumPy, Scikit-learn, and TensorFlow or PyTorch.
  Proficiency in version control systems, particularly Git, and experience with collaborative software development workflows.
  Strong background in statistical methods including hypothesis testing, regression analysis, time-series modeling, and Bayesian inference.
  Strong business acumen and demonstrated ability to translate analytical insights into strategic business recommendations.
  Experience with big data technologies such as Apache Spark, Hadoop, or Databricks at scale.
  Bachelor's or Master's degree in Computer Science, Statistics, Mathematics, Data Science, or a related quantitative field; PhD strongly preferred.
  Experience analyzing complex operational or people data (e.g., customer behavior, financial metrics, operational KPIs).
  Proven ability to communicate technical concepts clearly and persuasively to diverse audiences, including senior executives.

Preferred:
  Functional knowledge of modern data warehouse platforms such as Snowflake, BigQuery, or Redshift.
  Prior experience in a management consulting or professional services environment.
  Experience with geospatial data analysis and visualization.
  Working knowledge of HR fundamentals including talent acquisition, performance management, and organizational effectiveness.
  Familiarity with large language models (LLMs), prompt engineering, and retrieval-augmented generation (RAG) techniques.

Our Benefits Package Includes:
  Home office stipend for remote employees to set up a productive and comfortable work environment.
  Generous paid time off policy, including vacation days, sick leave, personal days, and company-observed holidays.
  Competitive base salary commensurate with experience, plus an annual performance-based bonus.
  Tuition reimbursement program for employees pursuing continuing education or advanced degrees.
  Annual professional development budget of up to $3,000 for conferences, online courses, books, and certifications.
  Collaborative, inclusive, and diverse workplace culture with active employee resource groups.
  Monthly wellness reimbursement for gym memberships, mental health apps, fitness equipment, and related expenses.
  401(k) or 403(b) retirement savings plan with competitive employer matching contributions.

Pinnacle AI Labs celebrates diversity and is committed to building an equitable and inclusive workplace.  All qualified applicants will receive consideration for employment without regard to race, color, religion, sex, sexual orientation, gender identity, national origin, or protected veteran status.

Ready to make an impact?  Apply now and join a team that is passionate about using data to change the world."""

score, classification = detect_ai(input, ai_words, ai_bigrams, ai_trigrams)
print(f"AI Score: {score}, Verdict: {classification}")

AI Score: 0.3204, Verdict: AI


In [ ]:
# this is from the scraped dataset


input = """Data AnalystAssist in data acquisition, validation, analysis, and reporting of key areas company wide. Provide support in development, testing, and implementation of processes to automate analysis and risk identification. Leverage technology to identify areas for process efficiency improvements while providing insights and possible recommendations.Compensation & Benefits:Estimated Starting Salary Range for Data Analyst : $95K-$110K.Pay commensurate with experience.Full time benefits include Medical, Dental, Vision, 401K and other possible benefits as provided. Benefits are subject to change with or without notice.Data Analyst Responsibilities Include:

Assists with development and maintenance of queries; performs data analytics to isolate anomalies, trends, fraudulent activities and/or to draw conclusions on objectives, within assigned time and budget.
Utilizes tools for data visualization, modeling, data management, and dynamic reporting.
Participates in development of scope, objectives, and analysis.
Integrates with assurance team on projects, including investigations and continuous assurance, at all phases to deliver data analytics and data requests.
Analyzes processes, transactional data and compliance with laws, regulations, and company policies, while documenting results.
Documents results based on defined standards; communicates results to clients and management.
Reports continuous assurance related outputs to management.
Performs other job-related duties as assigned.

Data Analyst Experience, Education, Skills, Abilities requested:

Bachelor‚Äôs Degree in a business-related field, computer science or information systems and 2 to 5 years of data analysis or data science experience or an equivalent combination of education and experience.
Data analysis scripting (SQL) and programming experience preferred. Certifications or recorded training in data analytics, business analytics, auditing, or process improvement preferred.
Accountability - Understanding the importance of taking personal responsibility; ability to take personal responsibility for assignments, decisions and results and focus on activities that have the greatest impact on meeting work commitments.
Accuracy and Attention to Detail - Understanding the necessity and value of accuracy; ability to complete tasks with high levels of precision.
Analytical Thinking - Knowledge of techniques and tools that promote effective analysis; ability to determine the root cause of organizational problems and create alternative solutions that resolve these problems.
Creativity - Knowledge of the approaches, tools, and techniques for promoting creative, original thinking and ability to apply it to a variety of business situations.
Data Gathering and Reporting - Knowledge of tools, techniques, and processes for gathering and reporting data; ability to practice them in a particular department or division of a company.
Decision Making and Critical Thinking - Knowledge of the decision-making process and associated tools and techniques; ability to accurately analyze situations and reach productive decisions based on informed judgment.
Effective Communications - Understanding of effective communication concepts, tools, and techniques; ability to effectively transmit, receive and accurately interpret ideas, information and needs through the application of appropriate communication behaviors.
Flexibility and Adaptability - Knowledge of successful approaches and techniques for dealing with change; ability to adapt to a changing environment and be comfortable with change.
Initiative - Being proactive and committing to action on self-identified job responsibilities and challenges; ability to seek out work and the drive to accomplish goals.
Internal Controls - Knowledge of concept, methods, and processes of internal control; ability to create, implement, evaluate, and enhance processes in internal controls.
Interpersonal Relationships - Knowledge of the techniques and the ability to work with a variety of individuals and groups in a constructive and collaborative manner.
Managing Multiple Priorities - Knowledge of effective self-management practices; ability to manage multiple concurrent objectives, projects, groups, or activities, making effective judgments as to prioritizing and time allocation.
Problem Solving - Knowledge of approaches, tools, techniques for recognizing, anticipating, and resolving organizational, operational or process problems; ability to apply knowledge of problem solving appropriately to diverse situations.
Teamwork - Knowledge of the necessity and value of teamwork; experience with; ability to work cooperatively towards shared goals and being supportive of others at all levels.
Past applicable job experience may include, but is not limited to: Business Analyst, Data Engineer, and Data Consultant.
Must pass pre-employment qualifications of Cherokee Federal

Company Information:Cherokee Nation Management and Consulting (CNMC) provides support, services, and solutions to federal and commercial customers. The company takes a personalized approach to solving our clients' toughest challenges, helping you make the most of your skills. CNMC is part of Cherokee Federal ‚Äì a team of tribally owned federal contracting companies. For more information, visit cherokee-federal.com.Please Note: This position is pending a contract award.If you are interested in a future with Cherokee Federal, APPLY TODAY! Although this is not an approved position, we are accepting applications for this future and anticipated need.#CherokeeFederal #LISimilar searchable job titlesBusiness AnalystData EngineerData ArchitectData ConsultantBusiness Intelligence AnalystKeywordsData visualizationAnalysisStatistical analysisData miningData-driven insightsLegal Disclaimer: Cherokee Federal is an equal opportunity employer. Please visit cherokee-federal.com/careers for information regarding our Affirmative Action and Equal Opportunity Employer Statement, Accommodation request, and Presidential EO 14042 Notice.
Job Type: Full-time
Pay: $95,000.00 - $110,000.00 per year
Benefits:

401(k) matching
Dental insurance
Health insurance
Paid time off
Parental leave
Vision insurance

Experience level:

2 years

Schedule:

8 hour shift

Work Location: In person"""

score, classification = detect_ai(input, ai_words, ai_bigrams, ai_trigrams)
print(f"AI Score: {score}, Verdict: {classification}")

AI Score: 0.1527, Verdict: Human
